# HSQC-style offset plots for small / median / large corr_rmse pairs

Run after `binding_vs_corr_rmse.ipynb`. Picks three pairs from `decomposition_no_othernoise.json` (`n_common > 100`) and draws the N/H shifts of their shared residues.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

DECOMP = Path('../../data/analysis/Fig3/inherent_test/decomposition_no_othernoise.json')
BMRB   = Path('../../data/rBMRB/bmrb_cs_data_withoffset_noref_v1.json')

decomp = json.load(open(DECOMP))
bmrb   = json.load(open(BMRB))

# entry_ID maps the internal numeric key -> eid (e.g. 'bmr17477_1'). Build the reverse lookup.
eid2key = {v: k for k, v in bmrb['entry_ID'].items()}
print(f'{len(decomp)} decomposition records, {len(eid2key)} BMRB entries')

In [ ]:
# --- Select small / medium / large corr_rmse pairs (n_common > 100) ---
cand = [r for r in decomp if r.get('corr_rmse') is not None]
cand.sort(key=lambda r: r['corr_rmse'])

import pandas as pd

cols = ['eid_a', 'eid_b', 'method', 'n_common', 'corr_rmse', 'total_rmse', 'offset', 'oa', 'ob']
df = pd.DataFrame(cand)[cols]          # cand is already sorted by corr_rmse
df.index.name = 'cand_idx'             # same index you use in cand[...] for picks

pd.set_option('display.max_rows', None, 'display.width', 200)
print(df.round(3).to_string())

for r in cand:
    r['offset_rmse'] = r['offset_mse'] ** 0.5


picks = {
    'small':  cand[0],
    'medium': cand[len(cand)//2 - 1],
    'large':  cand[-5],
}

# print(cand[-7])

for label, r in picks.items():
    print(f"{label:6s}  corr_rmse={r['corr_rmse']:.3f}  n_common={r['n_common']:>4d}  "
          f"{r['eid_a']} vs {r['eid_b']} ({r['method']})")


In [ ]:
def matched_NH(eid_a, eid_b):
    """Return (residue_idx, Na, Ha, Nb, Hb) arrays for residues where BOTH entries
    have non-null N and H shifts."""
    ka, kb = eid2key[eid_a], eid2key[eid_b]
    Na, Ha = bmrb['N'][ka], bmrb['H'][ka]
    Nb, Hb = bmrb['N'][kb], bmrb['H'][kb]
    n = min(len(Na), len(Ha), len(Nb), len(Hb))
    res, na, ha, nb, hb = [], [], [], [], []
    for i in range(n):
        va = (Na[i], Ha[i]); vb = (Nb[i], Hb[i])
        if None in va or None in vb:
            continue
        res.append(i); na.append(Na[i]); ha.append(Ha[i]); nb.append(Nb[i]); hb.append(Hb[i])
    return (np.array(res), np.array(na), np.array(ha), np.array(nb), np.array(hb))

In [ ]:
def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,                         
        "font.size": base,
        "axes.titlesize": base,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=16)
def plot_hsqc_offset(ax, rec, label, apply_offset=True, reason=None):
    """HSQC-style plot: 1H (x) vs 15N (y), both axes inverted.
    Each matched residue is drawn for both entries and connected by a line
    showing the peak displacement between the two datasets.

    apply_offset=True: correct the N values with each entry's own offset (oa/ob) before plotting
    (corrected = raw + offset, N only), so the displacements shown correspond to
    corr_rmse (inherent, offset removed).
    apply_offset=False: plot the raw N/H values; displacements correspond to total_rmse.
    """
    res, na, ha, nb, hb = matched_NH(rec['eid_a'], rec['eid_b'])

    if apply_offset:
        na = na + rec['oa']
        nb = nb + rec['ob']

    # ---- adjustable parameters ----
    LINE_COLOR = '#7f7f7f'   # connecting-line colour
    LINE_WIDTH = 1.2         # connecting-line width
    SIZE_A     = 80          # marker size, entry_a (open circles)
    SIZE_B     = 50          # marker size, entry_b (filled dots)

    # connecting segments, single fixed color, no arrows
    segs = np.stack([np.column_stack([ha, na]), np.column_stack([hb, nb])], axis=1)
    lc = LineCollection(segs, colors=LINE_COLOR, linewidths=LINE_WIDTH, alpha=0.9, zorder=2)
    ax.add_collection(lc)

    # peaks: entry_a hollow, entry_b filled
    ax.scatter(ha, na, s=SIZE_A, facecolors='none', edgecolors='#1f3b73',
               linewidths=1.1, zorder=3, label=rec['eid_a'])
    ax.scatter(hb, nb, s=SIZE_B, color='#61AFEF', zorder=4, label=rec['eid_b'], alpha=0.8)

    ax.set_xlabel('$^{1}$H (ppm)')
    ax.set_ylabel('$^{15}$N (ppm)')
    ax.invert_xaxis(); ax.invert_yaxis()  # HSQC convention

    rmse_shown = rec['corr_rmse'] if apply_offset else rec['total_rmse']
    rmse_label = 'Intrinsic RMSE' if apply_offset else 'total_rmse (raw, offset included)'
    # ax.set_title(f"{label}  |  {rmse_label} = {rmse_shown:.3f}\n"
    #             f"{rec['eid_a']}  vs  {rec['eid_b']}  (n = {len(res)}, {rec['method']})")
    ax.set_title(f"{label.upper()}  |  {rmse_label} = {rmse_shown:.3f}")
    ax.legend(loc='best', framealpha=0.5)
    # ax.grid(True, ls=':', alpha=0.4)

In [ ]:
for label in ['small', 'medium', 'large']:
    fig, ax = plt.subplots(figsize=(6.5, 5.2))
    plot_hsqc_offset(ax, picks[label], label)
    fig.tight_layout()
    Path('Figs').mkdir(exist_ok=True)
    fig.savefig(f'Figs/hsqc_offset_{label}.svg', format='svg')
    fig.savefig(f'Figs/hsqc_offset_{label}.png', format='png', dpi=300)
    plt.show()